In [4]:
import json
import os

from dotenv import load_dotenv
import httpx
import openai
from openai.types.chat import ChatCompletionMessage

load_dotenv()

MODEL = "gpt-4o-mini"
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

SYSTEM_PROMPT = """
        You are a Movie Expert agent backed by the Nomad Movies API.

    Tools:
    - get_popular_movies — list popular movies (GET /movies). Use when the user asks what is popular or trending now.
    - get_movie_details — details for one movie by numeric id (GET /movies/:id). Use when the user asks about a specific movie by ID.
    - get_movie_credits — cast and crew (GET /movies/:id/credits). Use when the user asks who stars, who directed, or credits for a movie ID.

    Call the appropriate tool to fetch data. After you receive tool results, answer the user in natural language using that data. Match the user’s language (e.g. Korean if they wrote in Korean).
    """

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Fetch popular movies from the /movies endpoint.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Fetch movie information for a given movie id.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "Movie id."}},
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Fetch cast and crew for a given movie id.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "Movie id."}},
                "required": ["id"],
            },
        },
    },
]


def get_popular_movies() -> object:
    r = httpx.get(f"{BASE_URL}/movies", timeout=5.0)
    r.raise_for_status()
    return r.json()


def get_movie_details(movie_id: int) -> object:
    r = httpx.get(f"{BASE_URL}/movies/{movie_id}", timeout=5.0)
    r.raise_for_status()
    return r.json()


def get_movie_credits(movie_id: int) -> object:
    r = httpx.get(f"{BASE_URL}/movies/{movie_id}/credits", timeout=5.0)
    r.raise_for_status()
    return r.json()


def execute_tool(name: str, arguments_json: str) -> str:
    try:
        args = json.loads(arguments_json)
        if name == "get_popular_movies":
            data = get_popular_movies()
        elif name == "get_movie_details":
            data = get_movie_details(int(args["id"]))
        elif name == "get_movie_credits":
            data = get_movie_credits(int(args["id"]))
        else:
            return json.dumps({"error": f"unknown tool: {name}"})
        return json.dumps(data)
    except Exception as e:
        return json.dumps({"error": type(e).__name__, "message": str(e)})


def _assistant_message_dict(msg: ChatCompletionMessage) -> dict:
    d: dict = {"role": "assistant"}
    if msg.content:
        d["content"] = msg.content
    if msg.tool_calls:
        d["tool_calls"] = [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in msg.tool_calls
        ]
    return d


def run_agent(user_content: str, verbose: bool = False) -> str:
    client = openai.OpenAI()
    messages: list = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
    )
    msg = resp.choices[0].message
    if not msg.tool_calls:
        return msg.content or ""

    if verbose:
        for tc in msg.tool_calls:
            print(f"  [tool] {tc.function.name} {tc.function.arguments}")

    messages.append(_assistant_message_dict(msg))
    for tc in msg.tool_calls:
        out = execute_tool(tc.function.name, tc.function.arguments)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})

    resp2 = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="none",
    )
    return resp2.choices[0].message.content or ""

In [5]:
TEST_PROMPTS = [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘",
]

for i, prompt in enumerate(TEST_PROMPTS, start=1):
    print(f"\n--- Test {i} ---")
    print(f"user: {prompt}")
    answer = run_agent(prompt, verbose=True)
    print(f"assistant: {answer}")


--- Test 1 ---
user: 지금 인기 있는 영화가 무엇인지 알려줘
  [tool] get_popular_movies {}
assistant: 현재 인기 있는 영화 목록입니다:

1. **내 마음이 부서질 거예요 (Your Heart Will Be Broken)** 
   - 개봉일: 2026년 3월 26일
   - ⭐ 평점: 6.6
   - 🎬 개요: 고등학생 폴리나는 새로운 학교에서 괴롭힘을 피하기 위해 괴롭히는 고등학생 바르스와 계약을 체결합니다. 그들은 서로의 감정이 자라나게 되는데...
   - ![포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

2. **아바타: 불과 재 (Avatar: Fire and Ash)**
   - 개봉일: 2025년 12월 17일
   - ⭐ 평점: 7.4
   - 🎬 개요: 제이크 설리와 네이티리는 새로운 위협인 '재인들'과 싸우기 위해 가족과 함께 힘을 합쳐야 합니다.
   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **호퍼스 (Hoppers)**
   - 개봉일: 2026년 3월 4일
   - ⭐ 평점: 7.6
   - 🎬 개요: 과학자들은 인간의 의식을 로봇 동물로 옮길 수 있는 방법을 발견하고, 동물과 소통할 수 있는 기회를 가진 주인공이 동물 세계의 신비를 밝혀냅니다.
   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **범죄 101 (Crime 101)**
   - 개봉일: 2026년 2월 11일
   - ⭐ 평점: 7.1
   - 🎬 개요: 한 도둑이 마지막 범죄로 큰 기회를 겨냥하는데, 그 과정에서 불미스러운 사건들이 이어집니다.
   - ![포스터](https://image.tmdb.org/t/p/w780/tVvpFIoteRHNn